In [20]:
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt
import os

os.makedirs('data/processed', exist_ok=True)

In [21]:
# фильтр низких частот, убираем шум выше 3 Гц
def butter_filter(data, cutoff=3.0, fs=20.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return filtfilt(b, a, data)

In [22]:
# нарезаем данные на окна по 60 точек (3 секунды) с перекрытием 50%
def create_windows(df, window_size=60, overlap=30):
    segments = []
    # ищем непрерывные блоки с одинаковой активностью
    df['block'] = (df['Activity'] != df['Activity'].shift()).cumsum()
    
    for _, group in df.groupby('block'):
        if len(group) < window_size:
            continue
        
        x_filt = butter_filter(group['acc_x'].values)
        y_filt = butter_filter(group['acc_y'].values)
        z_filt = butter_filter(group['acc_z'].values)
        
        start = 0
        while start + window_size <= len(group):
            segments.append({
                'Activity': group['Activity'].iloc[0],
                'x': x_filt[start:start + window_size],
                'y': y_filt[start:start + window_size],
                'z': z_filt[start:start + window_size]
            })
            start += (window_size - overlap)
    return pd.DataFrame(segments)

In [23]:
# считаем признаки для одного сигнала (временные и частотные)
def get_features(signal, fs=20.0):
    zcr = len(np.where(np.diff(np.sign(signal)))[0]) / len(signal)
    feats = {
        'mean': np.mean(signal), 'var': np.var(signal),
        'max': np.max(signal), 'min': np.min(signal),
        'energy': np.sum(signal**2), 'zcr': zcr
    }
    
    fft_vals = np.fft.rfft(signal)
    fft_freqs = np.fft.rfftfreq(len(signal), 1.0/fs)
    magnitudes = np.abs(fft_vals)
    
    feats['spectral_energy'] = np.sum(magnitudes**2)
    dom_freq = fft_freqs[np.argmax(magnitudes[1:]) + 1] if len(magnitudes) > 1 else 0.0
    feats['dominant_freq'] = dom_freq
    
    return feats

# извлекаем признаки для всех окон
def extract_features(df_seg):
    features_list = []
    for _, row in df_seg.iterrows():
        feats = {'Activity': row['Activity']}
        for axis in ['x', 'y', 'z']:
            axis_feats = get_features(row[axis])
            for k, v in axis_feats.items():
                feats[f'{axis}_{k}'] = v
        features_list.append(feats)
    return pd.DataFrame(features_list)

In [24]:
# основная функция обработки файла
def process_file(input_path, output_path):
    print(f"Processing {input_path}...")
    df = pd.read_csv(input_path, low_memory=False)
    
    df_seg = create_windows(df, window_size=60, overlap=30)
    print(f"  Windows: {len(df_seg)}")
    
    df_feat = extract_features(df_seg)
    df_feat.to_csv(output_path, index=False)
    print(f"  Saved: {output_path} ({df_feat.shape})\n")

# запускаем для train и test
process_file('data/raw/train.csv', 'data/processed/features_train.csv')
process_file('data/raw/test.csv', 'data/processed/features_test.csv')

Processing data/raw/train.csv...
  Windows: 28234
  Saved: data/processed/features_train.csv ((28234, 25))

Processing data/raw/test.csv...
  Windows: 7403
  Saved: data/processed/features_test.csv ((7403, 25))

